In [ ]:
import subprocess
from pathlib import Path

MAGICK = "magick"

ROOT = Path.cwd()

ASSETS = ROOT / "assets/banner"

OUTPUT = ROOT / "banner_output"

# Target banner size. Every background image is scaled to fully cover this
# canvas (cropping any overflow) regardless of its original dimensions/ratio.
CANVAS_WIDTH = 1920
CANVAS_HEIGHT = 1080

OUTPUT.mkdir(exist_ok=True)

# --------------------------------------------------------------------
# Layout (all coordinates live here)
# --------------------------------------------------------------------

LAYOUT = {
    "avatar": {
        "x": 110,
        "y": 412,
        "size": 275,
    },
    "name": {
        "x": 120,
        "y": 715,
        "font": ASSETS / "fonts" / "Resmont-Bold.ttf",
        "size": 36,
        "color": "#123B63",
    },
    "title": {
        "x": 120,
        "y": 760,
        "font": ASSETS / "fonts" / "Resmont-RegularItalic.otf",
        "size": 20,
        "color": "#294D73",
    },
}

In [ ]:
# --------------------------------------------------------------------
# Sessions
# --------------------------------------------------------------------

SESSIONS = [
    {
        "slug": "placeholder-slug",
        "bg": ROOT / "assets/banner/overlays/layout.png",
        "speaker_photo": ROOT / "assets/banner/speaker_images/speaker.jpg",
        "name": "Mr. Clark",
        "title": "Model at ScenePy",
    },

    # duplicate the part above to create a banner for other speakers too
]


In [ ]:
# --------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------

def run(cmd):
    print("\nRunning:")
    print(" ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)


def cover_canvas(image_path):
    """
    Wrap a background image in a magick MPR-style parenthetical group that
    scales it to cover CANVAS_WIDTH x CANVAS_HEIGHT (cropping overflow,
    centered) no matter its original size or aspect ratio.
    """
    return [
        "(",
        image_path,
        "-resize", f"{CANVAS_WIDTH}x{CANVAS_HEIGHT}^",
        "-gravity", "center",
        "-extent", f"{CANVAS_WIDTH}x{CANVAS_HEIGHT}",
        ")",
    ]


# --------------------------------------------------------------------
# Intro banner
# --------------------------------------------------------------------

def build_intro_banner(background, output):
    run([
        MAGICK,

        *cover_canvas(background),

        INTRO_OVERLAY,
        "-compose", "over",
        "-composite",
        output
    ])

In [ ]:
# --------------------------------------------------------------------
# Speaker banner
# --------------------------------------------------------------------

def build_banner(
        background,
        speaker_photo,
        speaker_name,
        speaker_title,
        output
):

    avatar = LAYOUT["avatar"]
    name = LAYOUT["name"]
    title = LAYOUT["title"]

    size = avatar["size"]

    run([
        MAGICK,

        *cover_canvas(background),

        "(",

            speaker_photo,

            "-resize", f"{size}x{size}^",
            "-gravity", "center",
            "-extent", f"{size}x{size}",

            "(",
                "-size", f"{size}x{size}",
                "xc:none",
                "-fill", "white",
                "-draw",
                f"circle {size//2},{size//2} {size//2},1",
                "-background", "black",
                "-alpha", "remove",
                "-alpha", "off",
            ")",

            "-compose", "copyopacity",
            "-composite",

        ")",

        "-gravity", "NorthWest",
        
        "-geometry",
        f"+{avatar['x']}+{avatar['y']}",

        "-compose", "over",
        "-composite",

        "-gravity", "NorthWest",
        
        "-font", name["font"],
        "-fill", name["color"],
        "-pointsize", str(name["size"]),
        "-annotate",
        f"+{name['x']}+{name['y']}",
        speaker_name,
        
        "-font", title["font"],
        "-fill", title["color"],
        "-pointsize", str(title["size"]),
        "-annotate",
        f"+{title['x']}+{title['y']}",
        speaker_title,

        output
    ])

In [ ]:
# --------------------------------------------------------------------
# Main
# --------------------------------------------------------------------
for session in SESSIONS:

    print(f"Generating {session['slug']}")

    build_banner(
        background=session["bg"],
        speaker_photo=session["speaker_photo"],
        speaker_name=session["name"],
        speaker_title=session["title"],
        output=OUTPUT / f"{session['slug']}.png",
    )